# XNES vs CMA-ES on COCO BBOB

This notebook runs two optimizers on the full COCO `bbob` suite with COCO-style budgeting.

- CMA-ES: restarting `cma.CMAEvolutionStrategy` with the same population size heuristic as xNES
- XNES: `Optimizer(schema, population_size=..., minimize=True)` with a mapping schema built from `problem.initial_solution`
- Budget per problem instance: `budget_multiplier * dimension`


In [1]:
from __future__ import annotations

from pathlib import Path
import shutil
import warnings

import cma
import cocoex
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from leitwerk import Optimizer, OptimizerSettings, Parameter
from leitwerk.xnes import _default_sample_count

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='The number of solutions passed to `tell` should.*')


In [2]:
# COCO experiment settings (algorithm settings remain defaults).
for output_dir in (Path('exdata'), Path('ppdata')):
    if output_dir.exists():
        shutil.rmtree(output_dir)

SUITE_NAME = 'bbob-noisy'
SUITE_INSTANCE = ''
SUITE_OPTIONS = ''
BUDGET_MULTIPLIER = 1000
SIGMA0_CMA = 1.0  # required constructor arg for CMA-ES

OUTPUT_ROOT = Path('exdata')  # COCO writes under this root automatically


In [3]:
def _vector_schema(x0: np.ndarray) -> tuple[list[str], dict[str, Parameter]]:
    names = [f'x{i:04d}' for i in range(x0.size)]
    schema = {name: Parameter(loc=float(value)) for name, value in zip(names, x0, strict=True)}
    return names, schema


def _population_size(dim: int) -> int:
    return _default_sample_count(None, dim)


def run_xnes(problem: cocoex.Problem, budget: int) -> None:
    x0 = np.asarray(problem.initial_solution, dtype=float)
    names, schema = _vector_schema(x0)
    population_size = _population_size(problem.dimension)
    settings = OptimizerSettings(
        population_size=population_size,
        minimize=True,
    )
    optimizer = Optimizer(schema, settings)

    while problem.evaluations < budget and not problem.final_target_hit:
        params = optimizer.ask()
        x = np.array([params[name] for name in names], dtype=float)
        value = float(problem(x))
        optimizer.tell(value)


def run_cma(problem: cocoex.Problem, budget: int, sigma0: float = SIGMA0_CMA) -> None:
    x_start = np.asarray(problem.initial_solution, dtype=float)
    population_size = _population_size(problem.dimension)

    while budget - problem.evaluations >= 3 and not problem.final_target_hit:
        es = cma.CMAEvolutionStrategy(
            x_start,
            sigma0,
            {'verb_disp': 0, 'verb_log': 0, 'popsize': population_size},
        )

        while budget - problem.evaluations >= 3 and not es.stop() and not problem.final_target_hit:
            remaining = budget - problem.evaluations
            candidates = es.ask(min(remaining, es.popsize))
            values = [float(problem(x)) for x in candidates]
            es.tell(candidates, values)

        x_start = np.asarray(es.result.xfavorite, dtype=float)


BENCHMARKS = {
    'cma': run_cma,
    'xnes': run_xnes,
}
ACTIVE_BENCHMARKS = [
    'cma',
    'xnes',
]  # comment out entries here to skip methods


In [4]:
def benchmark(label: str, runner) -> pd.DataFrame:
    suite = cocoex.Suite(SUITE_NAME, SUITE_INSTANCE, SUITE_OPTIONS)
    observer = cocoex.Observer(SUITE_NAME, f'result_folder: {label}')
    rows = []

    for problem in tqdm(suite, desc=label, unit='problem', dynamic_ncols=True):
        problem.observe_with(observer)
        budget = BUDGET_MULTIPLIER * problem.dimension
        start_evals = int(problem.evaluations)

        runner(problem, budget)

        rows.append({
            'optimizer': label,
            'problem_id': problem.id,
            'function': int(problem.id_function),
            'instance': int(problem.id_instance),
            'dimension': int(problem.dimension),
            'evaluations': int(problem.evaluations - start_evals),
            'budget': int(budget),
            'target_hit': bool(problem.final_target_hit),
            'best_f': float(problem.best_observed_fvalue1),
        })

        problem.free()

    suite.free()
    return pd.DataFrame(rows)


In [ ]:
results = pd.concat(
    [benchmark(label, BENCHMARKS[label]) for label in ACTIVE_BENCHMARKS],
    ignore_index=True,
)

results.head()


cma:   0%|          | 0/2700 [00:00<?, ?problem/s]

C:\Users\volke\PycharmProjects\leitwerk\.venv\Lib\site-packages\cma\utilities\utils.py:369: UserWarning: 
        geno-pheno transformation introduced based on the
        current covariance matrix with condition 1.4e+12 -> 1.0e+00,
        injected solutions become "invalid" in this iteration (time=Mar 17 05:21:23 2026 class=CMAEvolutionStrategy method=alleviate_conditioning iteration=262)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +
C:\Users\volke\PycharmProjects\leitwerk\.venv\Lib\site-packages\cma\utilities\utils.py:369: UserWarning: 
        geno-pheno transformation introduced based on the
        current covariance matrix with condition 1.0e+12 -> 1.0e+00,
        injected solutions become "invalid" in this iteration (time=Mar 17 05:21:34 2026 class=CMAEvolutionStrategy method=alleviate_conditioning iteration=300)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +
C:\Users\volke\PycharmProjects\leitwerk\.venv\Lib\site-packages\cma\utilities\utils.py

xnes:   0%|          | 0/2700 [00:00<?, ?problem/s]

In [ ]:
summary = (
    results.groupby(['optimizer', 'dimension'], as_index=False)
    .agg(
        success_rate=('target_hit', 'mean'),
        median_best_f=('best_f', 'median'),
        median_evals=('evaluations', 'median'),
        n=('problem_id', 'count'),
    )
)
summary


In [ ]:
pivot = summary.pivot(index='dimension', columns='optimizer', values='success_rate').sort_index()
ax = pivot.plot(marker='o', figsize=(8, 4), title='BBOB Success Rate by Dimension')
ax.set_ylabel('success rate')
ax.set_xlabel('dimension')
ax.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Optional COCO post-processing report (HTML)
import cocopp
cocopp.main([str(OUTPUT_ROOT / label) for label in ACTIVE_BENCHMARKS])
